# Distribution Shift Diagnosis

Analyze how model internals change between different inputs: activation
divergence at every hook, component vulnerability, feature stability,
layer adaptation difficulty, and prediction robustness.

## Why This Matters

This module provides tools for analyzing model internals. Understanding the internal mechanisms of transformer models is the core goal of mechanistic interpretability research — enabling us to move from treating models as black boxes to understanding the algorithms they implement.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.distribution_shift_diagnosis import (
    activation_divergence_profile,
    component_vulnerability,
    feature_stability,
    layer_adaptation_difficulty,
    prediction_robustness,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens_a = jnp.array([1, 15, 30, 45, 60, 75, 90])
tokens_b = jnp.array([5, 20, 35, 50, 65, 80, 95])
print('Model ready')

## Activation Divergence Profile

Measure how much activations differ between two inputs at every hook point.

In [ ]:
result = activation_divergence_profile(model, tokens_a, tokens_b)
print(f"Most divergent: {result['most_divergent']['hook']} (L2={result['most_divergent']['l2_distance']:.4f})")
print(f"Mean divergence: {result['mean_divergence']:.4f}\n")
for h in result['per_hook'][:8]:
    print(f"  {h['hook']:40s}  L2={h['l2_distance']:.4f}  cos={h['cosine_similarity']:.4f}")

## Component Vulnerability

Which heads and MLPs are most affected by the input shift?

In [ ]:
result = component_vulnerability(model, tokens_a, tokens_b)
print(f"Most vulnerable: {result['most_vulnerable']['component']}\n")
for c in result['per_component'][:10]:
    print(f"  {c['component']:8s}: abs_change={c['absolute_change']:.4f}, "
          f"rel_change={c['relative_change']:.4f}")

## Feature Stability

Track a specific direction's projection across layers for both inputs.

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = feature_stability(model, tokens_a, tokens_b, direction)
print(f"Stability fraction: {result['stability_fraction']:.2%}")
print(f"Most unstable layer: {result['most_unstable_layer']}\n")
for p in result['per_layer']:
    stable = 'stable' if p['stable'] else 'UNSTABLE'
    print(f"  Layer {p['layer']}: proj_a={p['projection_a']:+.4f}, proj_b={p['projection_b']:+.4f}, "
          f"diff={p['projection_diff']:.4f} [{stable}]")

## Layer Adaptation Difficulty

Which layers struggle most with the input shift?

In [ ]:
result = layer_adaptation_difficulty(model, tokens_a, tokens_b)
print(f"Hardest layer: {result['hardest_layer']}")
print(f"Mean difficulty: {result['mean_difficulty']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: L2={p['l2_distance']:.4f}, cos={p['cosine_similarity']:.4f}, "
          f"difficulty={p['difficulty']:.4f}")

## Prediction Robustness

How much do predictions change between the two inputs?

In [ ]:
result = prediction_robustness(model, tokens_a, tokens_b)
print(f"Agreement rate: {result['agreement_rate']:.2%}")
print(f"Mean KL divergence: {result['mean_kl_divergence']:.4f}\n")
for p in result['per_position']:
    agree = '=' if p['agree'] else '≠'
    print(f"  Pos {p['position']}: token_a={p['top_token_a']:3d} {agree} token_b={p['top_token_b']:3d}, "
          f"KL={p['kl_divergence']:.4f}")